# Scatter Plot - Data Analysis Notebook

In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt

In [2]:
# Read json files from ../src/json/fips.json
# FIPS 10-4 to Country Name mapping
with open('../src/json/fips.json', 'r') as f:
    fips = json.load(f)

In [3]:
# Function to convert 2-letter FIPS 10-4 country codes to country names
def code_to_country(code):
    try:
        country = fips[code]
        return country
    except:
        raise ValueError(f"Invalid country code: {code}")

In [4]:
# Load the mentions dataset from data/GDELT/weekly_media_attention_query_class34_better_disambiguation.csv
# The old dataset
# path_mentions = "../data/GDELT/weekly_media_attention_query_class34_better_disambiguation.csv"

# The new one
path_mentions = "../data/GDELT/2026_weekly_media_attention_query_class34_better_disambiguation.csv"

df_mentions = pd.read_csv(path_mentions)

# Display the first few rows of the dataframe
df_mentions.head()

,mention_week,conflict_country,media_country,mentions_count,distinct_events,verbal_conflict_mentions,material_conflict_mentions,verbal_conflict_unique_events,material_conflict_unique_events,state_mentions,...,civilians_unique_events,international_mentions,international_unique_events,avg_tone,stddev_tone,median_tone,avg_impact,stddev_impact,median_impact,top_article_url
0,2026-01-12,SO,Qatar,40,40,30,10,30,10,14,...,8,0,0,-3.171341,2.118406,-3.530633,-4.092500,2.276681,-4.0,https://arabic.arabianbusiness.com/arab-world-...
1,2026-01-12,SY,Lebanon,89,54,30,59,21,33,27,...,20,0,0,-5.492157,1.443785,-5.625000,-6.984270,2.825247,-7.2,https://www.hazard-herald.com/news/national/ni...
2,2026-01-12,SF,USA,9,9,3,6,3,6,6,...,2,0,0,-8.458729,2.266588,-10.081744,-7.333333,4.000000,-10.0,https://www.ksl.com:443/article/51431658/landm...
3,2026-01-12,FR,South Africa,13,13,10,3,10,3,4,...,2,3,3,-2.448098,2.758378,-1.459854,-5.115385,2.836913,-6.5,https://www.ewn.co.za/2026/01/15/musks-grok-ba...
4,2026-01-12,CO,Australia,16,4,16,0,4,0,16,...,0,0,0,-3.170181,0.014790,-3.180212,-4.400000,0.000000,-4.4,https://www.bunburymail.com.au/story/9152906/a...


In [5]:
# Print unique values in the 'mention_week' column sorted by most recent date
print(sorted(df_mentions['mention_week'].unique(), reverse=True))

['2026-01-12', '2026-01-05', '2025-12-29', '2025-12-22', '2025-12-15', '2025-12-08', '2025-12-01', '2025-11-24', '2025-11-17', '2025-11-10', '2025-11-03', '2025-10-27', '2025-10-20', '2025-10-13', '2025-10-06', '2025-09-29', '2025-09-22', '2025-09-15', '2025-09-08', '2025-09-01', '2025-08-25', '2025-08-18', '2025-08-11', '2025-08-04', '2025-07-28', '2025-07-21', '2025-07-14', '2025-07-07', '2025-06-30', '2025-06-09', '2025-06-02', '2025-05-26', '2025-05-19', '2025-05-12', '2025-05-05', '2025-04-28', '2025-04-21', '2025-04-14', '2025-04-07', '2025-03-31', '2025-03-24', '2025-03-17', '2025-03-10', '2025-03-03', '2025-02-24', '2025-02-17', '2025-02-10', '2025-02-03', '2025-01-27', '2025-01-20', '2025-01-13', '2025-01-06', '2024-12-30', '2024-12-23', '2024-12-16', '2024-12-09', '2024-12-02', '2024-11-25', '2024-11-18', '2024-11-11', '2024-11-04', '2024-10-28', '2024-10-21', '2024-10-14', '2024-10-07', '2024-09-30', '2024-09-23', '2024-09-16', '2024-09-09', '2024-09-02', '2024-08-26', '2024

In [6]:
# Since the ACLED dataset was updated the 12 of December 2025,
# we only keep GDELT data up to the 2025-12-08 week
df_mentions = df_mentions[df_mentions['mention_week'] <= '2025-12-08']

In [6]:
# Removing rows with 'media_country' as 'LatAm (Left-Wing Block)' or 'Pan-Africa', since they are not countries and their mentions are redundant
df_mentions = df_mentions[~df_mentions['media_country'].isin(['LatAm (Left-Wing Block)', 'Pan-Africa'])]

In [7]:
# Load the fatalities dataset from data/ACLED/number_of_reported_fatalities_by_country-year_as-of-12Dec2025.csv
path_fatalities = "../data/ACLED/number_of_reported_fatalities_by_country-year_as-of-12Dec2025.csv"
df_fatalities = pd.read_csv(path_fatalities, encoding='utf-8', sep=';')

# Display the first few rows of the dataframe
df_fatalities.head()

,COUNTRY,YEAR,FATALITIES
0,Afghanistan,2017,36360
1,Afghanistan,2018,42991
2,Afghanistan,2019,41419
3,Afghanistan,2020,30977
4,Afghanistan,2021,42425


Create a new dataset with country name, number of mentions, and number of fatalities for the scatter plot visualization.

The year will be also kept to allow visualizing the evolution over time.

Pay attention: here **we are considering all the conflict mentions**, not only material conflict ones.

In [8]:
# First, add column YEAR (column 'mention_week' contains the year at the start)
df_mentions_year = df_mentions.copy()
df_mentions_year['year'] = df_mentions_year['mention_week'].str[:4].astype(int)
# Remove entries with country codes not in fips keys
df_mentions_year = df_mentions_year[df_mentions_year['conflict_country'].isin(set(fips.keys()))]
# Aggregate the number of mentions by country and year
mentions_agg = df_mentions_year.groupby(['conflict_country', 'year'])['mentions_count'].sum().reset_index()
# Rename columns for clarity
mentions_agg.columns = ['COUNTRY', 'YEAR', 'MENTIONS']
# Map country codes to country names
mentions_agg['COUNTRY'] = mentions_agg['COUNTRY'].apply(code_to_country)
# If a country appears multiple times, sum the mentions
mentions_agg = mentions_agg.groupby(['COUNTRY', 'YEAR'])['MENTIONS'].sum().reset_index()

# Select relevant columns
fatalities_agg = df_fatalities[['COUNTRY', 'YEAR', 'FATALITIES']]
# Merge the two datasets on country name
merged_df = pd.merge(mentions_agg, fatalities_agg, on=['COUNTRY', 'YEAR'], how='inner')
# Display the merged dataframe
merged_df.head()

,COUNTRY,YEAR,MENTIONS,FATALITIES
0,Afghanistan,2017,1069744,36360
1,Afghanistan,2018,967731,42991
2,Afghanistan,2019,606170,41419
3,Afghanistan,2020,411984,30977
4,Afghanistan,2021,989289,42425


In [9]:
# Show highest values of mentions and fatalities
print("Top 10 countries by mentions in 2025:")
print(merged_df.sort_values(by='MENTIONS', ascending=False).head(10))
print()

print("Top 10 countries by fatalities in 2025:")
print(merged_df.sort_values(by='FATALITIES', ascending=False).head(10))
print()

print("Bottom 10 countries by mentions in 2025:")
print(merged_df.sort_values(by='MENTIONS', ascending=True).head(10))
print()

print("Bottom 10 countries by fatalities in 2025:")
print(merged_df.sort_values(by='FATALITIES', ascending=True).head(10))

Top 10 countries by mentions in 2025:
            COUNTRY  YEAR  MENTIONS  FATALITIES
1688  United States  2020  16416480          74
1689  United States  2021  13272503          87
1691  United States  2023  12518116          66
1692  United States  2024  12059771          33
1693  United States  2025  11266064          50
1690  United States  2022   9896049          98
780          Israel  2024   7102629          99
779          Israel  2023   5770340        1772
1334         Russia  2022   5329905          97
774          Israel  2018   4449792           7

Top 10 countries by fatalities in 2025:
          COUNTRY  YEAR  MENTIONS  FATALITIES
1671      Ukraine  2024   2212055       73570
1672      Ukraine  2025   1957967       73018
752          Iraq  2016   2035290       56032
1552        Syria  2017   2928484       54122
1     Afghanistan  2018    967731       42991
4     Afghanistan  2021    989289       42425
2     Afghanistan  2019    606170       41419
1669      Ukraine  2022  

In [10]:
# Save the merged dataset to a new CSV file
output_path = "../data/processed/scatter_plot.csv"
merged_df.to_csv(output_path, index=False)
print(f"Merged dataset saved to {output_path}")

Merged dataset saved to ../data/processed/scatter_plot.csv


### Taking only the material conflict mentions and not the verbal conflict ones

In [11]:
# Create a new dataset with country name, number of mentions in 2025, and number of fatalities in 2025

df_mentions_year = df_mentions.copy()
df_mentions_year['year'] = df_mentions_year['mention_week'].str[:4].astype(int)
df_mentions_year = df_mentions_year[df_mentions_year['conflict_country'].isin(set(fips.keys()))]
mentions_agg = df_mentions_year.groupby(['conflict_country', 'year'])['material_conflict_mentions'].sum().reset_index()
mentions_agg.columns = ['COUNTRY', 'YEAR', 'MENTIONS']
mentions_agg['COUNTRY'] = mentions_agg['COUNTRY'].apply(code_to_country)
mentions_agg = mentions_agg.groupby(['COUNTRY', 'YEAR'])['MENTIONS'].sum().reset_index()

# Select relevant columns
fatalities_agg = df_fatalities[['COUNTRY', 'YEAR', 'FATALITIES']]
# Merge the two datasets on country name
merged_df = pd.merge(mentions_agg, fatalities_agg, on=['COUNTRY', 'YEAR'], how='inner')
# Display the merged dataframe
merged_df.head()

,COUNTRY,YEAR,MENTIONS,FATALITIES
0,Afghanistan,2017,765896,36360
1,Afghanistan,2018,723215,42991
2,Afghanistan,2019,409893,41419
3,Afghanistan,2020,273208,30977
4,Afghanistan,2021,606561,42425


In [12]:
# Save the merged dataset to a new CSV file
output_path = "../data/processed/scatter_plot_material_conflict.csv"
merged_df.to_csv(output_path, index=False)
print(f"Merged dataset saved to {output_path}")

Merged dataset saved to ../data/processed/scatter_plot_material_conflict.csv
